# Data Preprocessing Pipeline

Input:
- (`licenses.csv`) NYC business license records containing business identifiers, license status, category, issuance and expiration dates, and location information.
- (`service_reqs.csv`) NYC 311 service request data, which records complaints types, timestamps, and geographic coordinates

Processing
- Clean and standardize key fields and convert into consistent formats
- Aggregate 311 complaints by business and month, and align them with active license periods
- Construct a monthly panel dataset for each business, engineer business-category indicators and complaint counts, and compute spatial clusters using location coordinates

Output
- A `joined_dataset.csv` containing business IDs, category features, complaint counts, license activity indicators, geographic clusters, and timestamps
- This dataset serves as the input for downstream modeling, incluing Cox survival analysis for predicting business closure risk.

The preprocessing and modeling pipeline was executed in Google Colab using a G4 GPU runtime, which accelerated computationally intensive steps such as large-scale data joins, feature construction, and model fitting.

### 1. Setup and load data

In [58]:
#!pip install -q sentence-transformers scikit-learn pandas

In [59]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import BallTree

LICENSES_PATH = "/content/drive/MyDrive/licenses.csv"
SERVICE_REQS_PATH = "/content/drive/MyDrive/service_reqs.csv"
OUTPUT_PATH = "/content/drive/MyDrive/joined_dataset.csv"

# convert 50m geo distance into radians for spatial clustering
LOCATION_K = 25
RADIUS_METERS = 50
EARTH_RADIUS_METERS = 6_371_000
RADIUS_RADIANS = RADIUS_METERS / EARTH_RADIUS_METERS

### 2. Read the source files

In [60]:
licenses = pd.read_csv(
    LICENSES_PATH,
    usecols=[
        "License Number",
        "Business Unique ID",
        "Business Category",
        "License Type",
        "License Status",
        "Initial Issuance Date",
        "Expiration Date",
        "Latitude",
        "Longitude",
    ],
)

service_reqs = pd.read_csv(
    SERVICE_REQS_PATH,
    usecols=[
        "Unique Key",
        "Created Date",
        "Problem (formerly Complaint Type)",
        "Latitude",
        "Longitude",
    ],
)

print("licenses shape:", licenses.shape)
print("service_reqs shape:", service_reqs.shape)


licenses shape: (32432, 9)
service_reqs shape: (4231052, 5)


### 3. Clean and standardize columns

In [61]:
licenses["Business Unique ID"] = licenses["Business Unique ID"].astype(str).str.strip()
licenses["License Number"] = licenses["License Number"].astype(str).str.strip()
licenses["Business Category"] = licenses["Business Category"].astype(str).str.strip()
licenses["License Type"] = licenses["License Type"].astype(str).str.strip()
licenses["License Status"] = licenses["License Status"].astype(str).str.strip()

licenses["Initial Issuance Date"] = pd.to_datetime(
    licenses["Initial Issuance Date"],
    format="%m/%d/%Y",
    errors="coerce",
)
licenses["Expiration Date"] = pd.to_datetime(
    licenses["Expiration Date"],
    format="%m/%d/%Y",
    errors="coerce",
)
licenses["Latitude"] = pd.to_numeric(licenses["Latitude"], errors="coerce")
licenses["Longitude"] = pd.to_numeric(licenses["Longitude"], errors="coerce")

service_reqs["Unique Key"] = service_reqs["Unique Key"].astype(str).str.strip()
service_reqs["Problem (formerly Complaint Type)"] = (
    service_reqs["Problem (formerly Complaint Type)"].astype(str).str.strip()
)
service_reqs["Created Date"] = pd.to_datetime(
    service_reqs["Created Date"],
    format="%m/%d/%Y %I:%M:%S %p",
    errors="coerce",
)
service_reqs["Latitude"] = pd.to_numeric(service_reqs["Latitude"], errors="coerce")
service_reqs["Longitude"] = pd.to_numeric(service_reqs["Longitude"], errors="coerce")

licenses = licenses.dropna(
    subset=[
        "Business Unique ID",
        "License Number",
        "Initial Issuance Date",
        "Expiration Date",
        "Business Category",
    ]
).copy()
licenses = licenses.loc[licenses["Business Category"] != ""].copy()

service_reqs = service_reqs.dropna(
    subset=[
        "Unique Key",
        "Created Date",
        "Latitude",
        "Longitude",
        "Problem (formerly Complaint Type)",
    ]
).copy()
service_reqs = service_reqs.loc[
    service_reqs["Problem (formerly Complaint Type)"] != ""
].copy()

licenses["issue_month"] = licenses["Initial Issuance Date"].dt.to_period("M").dt.to_timestamp()
licenses["expire_month"] = licenses["Expiration Date"].dt.to_period("M").dt.to_timestamp()
service_reqs["month"] = service_reqs["Created Date"].dt.to_period("M").dt.to_timestamp()

licenses = licenses.loc[licenses["expire_month"] >= licenses["issue_month"]].copy()

print("licenses after cleaning:", licenses.shape)
print("service_reqs after cleaning:", service_reqs.shape)

licenses after cleaning: (32404, 11)
service_reqs after cleaning: (4173100, 6)


### 4. Expand licenses into active business-month rows

Transforms each license record into a sequence of monthly observations representing the period during which the business was active.

In [62]:
def month_range(start, end):
    return pd.date_range(start=start, end=end, freq="MS")

licenses_expanded = licenses.copy()
licenses_expanded["month"] = licenses_expanded.apply(
    lambda row: month_range(row["issue_month"], row["expire_month"]), axis=1
)
licenses_expanded = licenses_expanded.explode("month").reset_index(drop=True)

print("expanded license-month rows:", licenses_expanded.shape)
licenses_expanded.head()

expanded license-month rows: (3897775, 12)


,License Number,Business Unique ID,Business Category,License Type,License Status,Initial Issuance Date,Expiration Date,Latitude,Longitude,issue_month,expire_month,month
0,0016371-DCA,BA-1322274-2022,Secondhand Dealer - General,Premises,Active,2003-10-31,2027-07-31,40.76249,-73.970125,2003-10-01,2027-07-01,2003-10-01
1,0016371-DCA,BA-1322274-2022,Secondhand Dealer - General,Premises,Active,2003-10-31,2027-07-31,40.76249,-73.970125,2003-10-01,2027-07-01,2003-11-01
2,0016371-DCA,BA-1322274-2022,Secondhand Dealer - General,Premises,Active,2003-10-31,2027-07-31,40.76249,-73.970125,2003-10-01,2027-07-01,2003-12-01
3,0016371-DCA,BA-1322274-2022,Secondhand Dealer - General,Premises,Active,2003-10-31,2027-07-31,40.76249,-73.970125,2003-10-01,2027-07-01,2004-01-01
4,0016371-DCA,BA-1322274-2022,Secondhand Dealer - General,Premises,Active,2003-10-31,2027-07-31,40.76249,-73.970125,2003-10-01,2027-07-01,2004-02-01


### 5. Build the monthly license-side panel with business categories

Creates the main license panel with one row per business-month and encodes business license categories into numeric indicator columns

In [63]:
first_license_dates = (
    licenses.groupby("Business Unique ID")["Initial Issuance Date"]
    .min()
    .rename("first_license_date")
    .reset_index()
)

license_counts = (
    licenses_expanded.groupby(["Business Unique ID", "month"])["License Number"]
    .nunique()
    .rename("active_license_count")
    .reset_index()
)

license_cat_counts = (
    licenses_expanded.groupby(
        ["Business Unique ID", "month", "Business Category"]
    )["License Number"]
    .count()
    .reset_index(name="count")
)

business_categories = sorted(license_cat_counts["Business Category"].dropna().unique())
business_category_to_idx = {name: idx for idx, name in enumerate(business_categories)}
license_cat_counts["business_category_idx"] = license_cat_counts["Business Category"].map(
    business_category_to_idx
)

license_cat_wide = license_cat_counts.pivot_table(
    index=["Business Unique ID", "month"],
    columns="business_category_idx",
    values="count",
    fill_value=0,
)

license_cat_wide.columns = [f"business_category_{int(c)}" for c in license_cat_wide.columns]
license_cat_wide = license_cat_wide.reset_index()

business_category_lookup = pd.DataFrame(
    {
        "business_category_idx": list(business_category_to_idx.values()),
        "original_business_category": list(business_category_to_idx.keys()),
    }
).sort_values("business_category_idx")

license_panel = (
    license_counts
    .merge(license_cat_wide, on=["Business Unique ID", "month"], how="left")
    .merge(first_license_dates, on="Business Unique ID", how="left")
    .rename(columns={"Business Unique ID": "business_id"})
)

license_panel["months_since_first_license"] = (
    (license_panel["month"].dt.year - license_panel["first_license_date"].dt.year) * 12
    + (license_panel["month"].dt.month - license_panel["first_license_date"].dt.month)
)
license_panel["open"] = 1

print("Unique business categories kept:", len(business_category_to_idx))
business_category_lookup.head()

Unique business categories kept: 48


,business_category_idx,original_business_category
0,0,Bingo Game Operator
1,1,Booting Company
2,2,Car Wash
3,3,Commercial Lessor - Bingo
4,4,Construction Labor Provider


### 6. Compute one representative location per business and cluster coordinates into regions

Determines a representative lat + lng for each business and groups nearby businesses into geographic location clusters.

In [64]:
business_locations = (
    licenses.dropna(subset=["Latitude", "Longitude"])
    .groupby("Business Unique ID")[["Latitude", "Longitude"]]
    .median()
    .reset_index()
    .rename(columns={"Business Unique ID": "business_id"})
)

print("Businesses with usable coordinates:", business_locations.shape[0])

if len(business_locations) >= LOCATION_K:
    kmeans = KMeans(n_clusters=LOCATION_K, random_state=42, n_init=20)
    business_locations["location_cluster"] = kmeans.fit_predict(
        business_locations[["Latitude", "Longitude"]]
    )
else:
    business_locations["location_cluster"] = 0

location_cluster_lookup = (
    business_locations.groupby("location_cluster")[["Latitude", "Longitude"]]
    .mean()
    .rename(
        columns={
            "Latitude": "cluster_center_lat",
            "Longitude": "cluster_center_lon",
        }
    )
    .reset_index()
    .sort_values("location_cluster")
)

business_locations.head()

Businesses with usable coordinates: 18546


,business_id,Latitude,Longitude,location_cluster
0,BA-1007652-2022,40.76389,-73.885927,24
1,BA-1010960-2022,40.72689,-73.853497,24
2,BA-1010972-2022,40.74013,-73.796261,11
3,BA-1010979-2022,40.67668,-73.860526,11
4,BA-1010980-2022,40.72604,-73.977960,9


### 7. Radius join 311 requests to businesses within 50 meters

Matches 311 complaints to businesses by identifying requests that occur within a 50-meter radius of each business location.

In [65]:
biz = business_locations.dropna(subset=["Latitude", "Longitude"]).copy()
biz["lat_rad"] = np.radians(biz["Latitude"])
biz["lon_rad"] = np.radians(biz["Longitude"])

req = service_reqs.dropna(subset=["Latitude", "Longitude"]).copy()
req["lat_rad"] = np.radians(req["Latitude"])
req["lon_rad"] = np.radians(req["Longitude"])

biz_coords = np.c_[biz["lat_rad"].values, biz["lon_rad"].values]
req_coords = np.c_[req["lat_rad"].values, req["lon_rad"].values]

tree = BallTree(biz_coords, metric="haversine")
indices_array = tree.query_radius(req_coords, r=RADIUS_RADIANS)

pairs = []
for req_idx, biz_indices in enumerate(indices_array):
    if len(biz_indices) == 0:
        continue
    req_row = req.iloc[req_idx]
    for biz_idx in biz_indices:
        biz_row = biz.iloc[biz_idx]
        pairs.append(
            {
                "Unique Key": req_row["Unique Key"],
                "month": req_row["month"],
                "complaint_type": req_row["Problem (formerly Complaint Type)"],
                "business_id": biz_row["business_id"],
            }
        )

req_business = pd.DataFrame(pairs)

print("Matched request-business pairs:", req_business.shape)
req_business.head()

Matched request-business pairs: (2277260, 4)


,Unique Key,month,complaint_type,business_id
0,68058349,2026-02-01,Illegal Parking,BA-1426961-2022
1,68051448,2026-02-01,Noise - Residential,BA-1556013-2022
2,68051448,2026-02-01,Noise - Residential,BA-1110049-2022
3,68051448,2026-02-01,Noise - Residential,BA-1319614-2022
4,68051448,2026-02-01,Noise - Residential,BA-1319744-2022


### 8. Aggregate monthly 311 counts with unclustered complaint types

Aggregates nearby 311 complaints by month and complaint type, creating numeric columns representing complaint activity for each business-month.

In [66]:
if len(req_business) == 0:
    complaint_panel = pd.DataFrame(columns=["business_id", "month", "total_311"])
    complaint_type_lookup = pd.DataFrame(
        columns=["complaint_type_idx", "original_complaint_type"]
    )
else:
    total_311 = (
        req_business.groupby(["business_id", "month"])["Unique Key"]
        .count()
        .rename("total_311")
        .reset_index()
    )

    complaint_counts = (
        req_business.groupby(["business_id", "month", "complaint_type"])["Unique Key"]
        .count()
        .reset_index(name="count")
    )

    complaint_types = sorted(complaint_counts["complaint_type"].dropna().unique())
    complaint_type_to_idx = {name: idx for idx, name in enumerate(complaint_types)}
    complaint_counts["complaint_type_idx"] = complaint_counts["complaint_type"].map(
        complaint_type_to_idx
    )

    complaint_wide = complaint_counts.pivot_table(
        index=["business_id", "month"],
        columns="complaint_type_idx",
        values="count",
        fill_value=0,
    )

    complaint_wide.columns = [f"complaint_type_{int(c)}" for c in complaint_wide.columns]
    complaint_wide = complaint_wide.reset_index()

    complaint_panel = total_311.merge(
        complaint_wide,
        on=["business_id", "month"],
        how="left",
    )

    complaint_type_lookup = pd.DataFrame(
        {
            "complaint_type_idx": list(complaint_type_to_idx.values()),
            "original_complaint_type": list(complaint_type_to_idx.keys()),
        }
    ).sort_values("complaint_type_idx")

print("Unique complaint types kept:", len(complaint_type_lookup))
complaint_panel.head()

Unique complaint types kept: 177


,business_id,month,total_311,complaint_type_0,complaint_type_1,complaint_type_2,complaint_type_3,complaint_type_4,complaint_type_5,complaint_type_6,...,complaint_type_167,complaint_type_168,complaint_type_169,complaint_type_170,complaint_type_171,complaint_type_172,complaint_type_173,complaint_type_174,complaint_type_175,complaint_type_176
0,BA-1007652-2022,2025-01-01,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BA-1007652-2022,2025-02-01,7,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BA-1007652-2022,2025-03-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BA-1007652-2022,2025-04-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BA-1007652-2022,2025-05-01,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 9. Merge license panel, complaint panel, and location cluster

Combines into single unified dataset.

In [67]:
joined = license_panel.merge(
    complaint_panel,
    on=["business_id", "month"],
    how="left",
)

joined = joined.merge(
    business_locations[["business_id", "location_cluster"]],
    on="business_id",
    how="left",
)

joined["total_311"] = joined["total_311"].fillna(0)

complaint_cols = [c for c in joined.columns if c.startswith("complaint_type_")]
for col in complaint_cols:
    joined[col] = joined[col].fillna(0)

joined["location_cluster"] = joined["location_cluster"].fillna(-1).astype(int)
joined.head()

,business_id,month,active_license_count,business_category_0,business_category_1,business_category_2,business_category_3,business_category_4,business_category_5,business_category_6,...,complaint_type_168,complaint_type_169,complaint_type_170,complaint_type_171,complaint_type_172,complaint_type_173,complaint_type_174,complaint_type_175,complaint_type_176,location_cluster
0,BA-1000527-2022,2023-09-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
1,BA-1000527-2022,2023-10-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
2,BA-1000527-2022,2023-11-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
3,BA-1000527-2022,2023-12-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1
4,BA-1000527-2022,2024-01-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1


### 10. Reorder columns and run sanity checks

Removes rows without a valid location cluster assignment.

In [68]:
business_cat_cols = sorted(
    [c for c in joined.columns if c.startswith("business_category_")],
    key=lambda x: int(x.split("_")[-1]),
)

complaint_cols = sorted(
    [c for c in joined.columns if c.startswith("complaint_type_")],
    key=lambda x: int(x.split("_")[-1]),
)

final_cols = (
    ["business_id", "month", "active_license_count"]
    + business_cat_cols
    + ["total_311"]
    + complaint_cols
    + ["open", "months_since_first_license", "location_cluster"]
)

joined = joined[final_cols].sort_values(["business_id", "month"]).reset_index(drop=True)

joined["business_category_sum"] = joined[business_cat_cols].sum(axis=1)
print(
    "Business category counts sum to active_license_count:",
    (joined["business_category_sum"] == joined["active_license_count"]).mean(),
)

if complaint_cols:
    joined["complaint_sum"] = joined[complaint_cols].sum(axis=1)
    print(
        "Complaint counts sum to total_311:",
        (joined["complaint_sum"] == joined["total_311"]).mean(),
    )

print("Number of rows in final panel:", len(joined))
print("Number of unique businesses:", joined["business_id"].nunique())
print("Month range:", joined["month"].min(), "to", joined["month"].max())

joined.head()

Business category counts sum to active_license_count: 1.0
Complaint counts sum to total_311: 1.0
Number of rows in final panel: 3636563
Number of unique businesses: 28933
Month range: 1900-12-01 00:00:00 to 2100-12-01 00:00:00


,business_id,month,active_license_count,business_category_0,business_category_1,business_category_2,business_category_3,business_category_4,business_category_5,business_category_6,...,complaint_type_172,complaint_type_173,complaint_type_174,complaint_type_175,complaint_type_176,open,months_since_first_license,location_cluster,business_category_sum,complaint_sum
0,BA-1000527-2022,2023-09-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,0,-1,1.0,0.0
1,BA-1000527-2022,2023-10-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,1,-1,1.0,0.0
2,BA-1000527-2022,2023-11-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,2,-1,1.0,0.0
3,BA-1000527-2022,2023-12-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,3,-1,1.0,0.0
4,BA-1000527-2022,2024-01-01,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1,4,-1,1.0,0.0


In [69]:
joined = joined[joined["location_cluster"] != -1].copy()

cluster_centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=["location_cluster_lat", "location_cluster_lng"]
)
cluster_centers["location_cluster"] = cluster_centers.index

joined = joined.merge(
    cluster_centers,
    on="location_cluster",
    how="left"
)

joined = joined[
    ~((joined["location_cluster_lat"] == 0) & (joined["location_cluster_lng"] == 0))
].copy()

print("Rows after filtering valid clusters and coordinates:", len(joined))
print("Unique businesses after filtering:", joined["business_id"].nunique())

Rows after filtering valid clusters and coordinates: 2240985
Unique businesses after filtering: 18542


In [70]:
missing_counts = joined.isna().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) == 0:
    print("No missing values in dataset.")
else:
    print("Columns with missing values:")
    print(missing_cols.sort_values(ascending=False))

print("Total missing values:", joined.isna().sum().sum())

No missing values in dataset.
Total missing values: 0


### 11. Save outputs

In [71]:
joined.to_csv(OUTPUT_PATH, index=False)

business_category_lookup.to_csv(
    "/content/drive/MyDrive/business_category_lookup_unclustered.csv", index=False
)
complaint_type_lookup.to_csv(
    "/content/drive/MyDrive/complaint_type_lookup_unclustered.csv", index=False
)
location_cluster_lookup.to_csv(
    "/content/drive/MyDrive/location_cluster_lookup.csv", index=False
)

print(f"Saved: {OUTPUT_PATH}")
print("Saved: /content/drive/MyDrive/business_category_lookup_unclustered.csv")
print("Saved: /content/drive/MyDrive/complaint_type_lookup_unclustered.csv")
print("Saved: /content/drive/MyDrive/location_cluster_lookup.csv")

Saved: /content/drive/MyDrive/joined_dataset.csv
Saved: /content/drive/MyDrive/business_category_lookup_unclustered.csv
Saved: /content/drive/MyDrive/complaint_type_lookup_unclustered.csv
Saved: /content/drive/MyDrive/location_cluster_lookup.csv
